To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://github.com/unslothai/unsloth?tab=readme-ov-file#-installation-instructions).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save) (eg for Llama.cpp).

[NEW] Supports all Qwen 2.5 model sizes! 0.5, 1.5, 3, 7, 14, 32, 72b!

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/drive/1T-YBVfnphoVc8E2E854qF3jdia2Ll2W2?usp=sharing)**

In [12]:
!pip install -q --upgrade unsloth

* We support Llama, Mistral, Phi-3, Gemma, Yi, DeepSeek, Qwen, TinyLlama, Vicuna, Open Hermes etc
* We support 16bit LoRA or 4bit QLoRA. Both 2x faster.
* `max_seq_length` can be set to anything, since we do automatic RoPE Scaling via [kaiokendev's](https://kaiokendev.github.io/til) method.
* [**NEW**] We make Gemma-2 9b / 27b **2x faster**! See our [Gemma-2 9b notebook](https://colab.research.google.com/drive/1vIrqH5uYDQwsJ4-OO3DErvuv4pBgVwk4?usp=sharing)
* [**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)
* [**NEW**] We make Mistral NeMo 12B 2x faster and fit in under 12GB of VRAM! [Mistral NeMo notebook](https://colab.research.google.com/drive/17d3U-CAIwzmbDRqbZ9NnpHxCkmXB6LZ0?usp=sharing)

In [13]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    model_name = "unsloth/Qwen2.5-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.8.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/drive/1XamvWYinY6FOSX9GLvnqSjjsNflxdhNc?usp=sharing).

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [16]:
from datasets import load_dataset

dataset = load_dataset(
    "GSMS-B/Indian-Legal-QA-BNS-BNSS-BSA",
    data_files="bns_bnss_bsa_combined_legal_qa.jsonl",
    split="train"
)

print(dataset)
print(dataset.column_names)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['chunk_id', 'act', 'section_number', 'section_title', 'question', 'answer', 'question_type'],
    num_rows: 6354
})
['chunk_id', 'act', 'section_number', 'section_title', 'question', 'answer', 'question_type']
{'chunk_id': 'BNS_1', 'act': 'BNS 2023', 'section_number': '1', 'section_title': 'Short title, commencement and application', 'question': 'Who is liable for punishment under the Indian criminal code for offences committed outside India?', 'answer': 'Any person liable under Indian law to be tried for an offence committed beyond India will be dealt with as if the act was committed within India. The law also applies to any citizen of India outside the country, any person on an Indian-registered ship or aircraft, and anyone committing an offence targeting an Indian computer resource from outside India. [Source: Section 1, BNS 2023]', 'question_type': 'definitional_topic'}


In [17]:
print(dataset)
print(dataset.column_names)
print(dataset[0])

Dataset({
    features: ['chunk_id', 'act', 'section_number', 'section_title', 'question', 'answer', 'question_type'],
    num_rows: 6354
})
['chunk_id', 'act', 'section_number', 'section_title', 'question', 'answer', 'question_type']
{'chunk_id': 'BNS_1', 'act': 'BNS 2023', 'section_number': '1', 'section_title': 'Short title, commencement and application', 'question': 'Who is liable for punishment under the Indian criminal code for offences committed outside India?', 'answer': 'Any person liable under Indian law to be tried for an offence committed beyond India will be dealt with as if the act was committed within India. The law also applies to any citizen of India outside the country, any person on an Indian-registered ship or aircraft, and anyone committing an offence targeting an Indian computer resource from outside India. [Source: Section 1, BNS 2023]', 'question_type': 'definitional_topic'}


In [18]:
alpaca_prompt = """Below is a question related to Indian criminal law.
Provide an accurate answer based on the relevant legal provisions.

### Question:
{}

### Answer:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    questions = examples["question"]
    answers = examples["answer"]

    texts = []

    for question, answer in zip(questions, answers):
        text = alpaca_prompt.format(
            question,
            answer
        ) + EOS_TOKEN

        texts.append(text)

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True
)

print(dataset[0]["text"])

Map:   0%|          | 0/6354 [00:00<?, ? examples/s]

Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
Who is liable for punishment under the Indian criminal code for offences committed outside India?

### Answer:
Any person liable under Indian law to be tried for an offence committed beyond India will be dealt with as if the act was committed within India. The law also applies to any citizen of India outside the country, any person on an Indian-registered ship or aircraft, and anyone committing an offence targeting an Indian computer resource from outside India. [Source: Section 1, BNS 2023]<|endoftext|>


In [19]:
print(dataset[0]["text"])

Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
Who is liable for punishment under the Indian criminal code for offences committed outside India?

### Answer:
Any person liable under Indian law to be tried for an offence committed beyond India will be dealt with as if the act was committed within India. The law also applies to any citizen of India outside the country, any person on an Indian-registered ship or aircraft, and anyone committing an offence targeting an Indian computer resource from outside India. [Source: Section 1, BNS 2023]<|endoftext|>


<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [20]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        warmup_steps=10,

        num_train_epochs=1,

        learning_rate=2e-4,

        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        logging_steps=50,

        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",

        seed=3407,

        output_dir="outputs",
        report_to="none",

        save_strategy="epoch",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6354 [00:00<?, ? examples/s]

In [21]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
7.242 GB of memory reserved.


In [22]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,354 | Num Epochs = 1 | Total steps = 795
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
50,1.542210
100,1.177329
150,1.133592
200,1.128271
250,1.109231
300,1.090912


Step,Training Loss
50,1.542210
100,1.177329
150,1.133592
200,1.128271
250,1.109231
300,1.090912
350,1.085709
400,1.076054
450,1.049952
500,1.060497


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-795/tokenizer_config.json.


In [23]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

3336.9208 seconds used for training.
55.62 minutes used for training.
Peak reserved memory = 7.982 GB.
Peak reserved memory for training = 0.74 GB.
Peak reserved memory % of max memory = 54.81 %.
Peak reserved memory for training % of max memory = 5.081 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/drive/1T-YBVfnphoVc8E2E854qF3jdia2Ll2W2?usp=sharing)**

In [24]:
FastLanguageModel.for_inference(model)

question = "What is the punishment for theft under the Bharatiya Nyaya Sanhita, 2023?"

prompt = alpaca_prompt.format(
    question,
    "",
    ""
)

inputs = tokenizer(
    [prompt],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    use_cache=True
)

response = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)

print(response[0])

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
What is the punishment for theft under the Bharatiya Nyaya Sanhita, 2023?

### Answer:
Theft is punishable by up to 3 years in prison, or a fine, or both. If the stolen property is movable and the offender is caught in possession of it, the offender shall be punished with imprisonment up to 1 year, or a fine, or both. [Source: Section 303, BNS 2023]


In [25]:
FastLanguageModel.for_inference(model)

test_questions = [
    "What is the punishment for theft under the Bharatiya Nyaya Sanhita, 2023?",

    "What is the punishment for voluntarily causing hurt under the Bharatiya Nyaya Sanhita, 2023?",

    "What is the purpose of Section 1 of the Bharatiya Nyaya Sanhita, 2023?",

    "Does the Bharatiya Nyaya Sanhita apply to offences committed outside India?",

    "What are the consequences of committing an offence against an Indian computer resource from outside India?"
]

for i, question in enumerate(test_questions, 1):

    prompt = alpaca_prompt.format(
        question,
        "",
        ""
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        use_cache=True
    )

    response = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    print("=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print(response)
    print()

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 1
Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
What is the punishment for theft under the Bharatiya Nyaya Sanhita, 2023?

### Answer:
Theft is punishable by up to 3 years in prison, or a fine, or both. If the stolen property is movable and the offender is caught in possession of it, the offender shall be punished with imprisonment up to 1 year, or a fine, or both. [Source: Section 303, BNS 2023]



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 2
Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
What is the punishment for voluntarily causing hurt under the Bharatiya Nyaya Sanhita, 2023?

### Answer:
Causing hurt is punishable by up to one year in prison, or a fine, or both. If the hurt is grievous, the punishment increases to up to three years in prison, or a fine, or both. [Source: Section 115, BNS 2023]



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 3
Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
What is the purpose of Section 1 of the Bharatiya Nyaya Sanhita, 2023?

### Answer:
Section 1 of the Bharatiya Nyaya Sanhita, 2023, establishes the fundamental principles of the Indian criminal justice system, including the presumption of innocence, the right to a fair trial, and the prohibition of cruel or degrading punishment. [Source: Section 1, BNS 2023]



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 4
Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
Does the Bharatiya Nyaya Sanhita apply to offences committed outside India?

### Answer:
No, the Sanhita does not extend to any offence committed outside India, unless the Central Government has made special provisions for it. [Source: Section 1, BNS 2023]

TEST 5
Below is a question related to Indian criminal law. 
Provide an accurate answer based on the relevant legal provisions.

### Question:
What are the consequences of committing an offence against an Indian computer resource from outside India?

### Answer:
Committing an offence against an Indian computer resource from outside India is punishable by up to seven years in prison and a fine. [Source: Section 249, BNS 2023]



 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [26]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


('lora_model/tokenizer_config.json', 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [28]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [29]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [30]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in `llama.cpp` or a UI based system like `GPT4All`. You can install GPT4All by going [here](https://gpt4all.io/index.html).

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/drive/1T-YBVfnphoVc8E2E854qF3jdia2Ll2W2?usp=sharing)**

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/u54VK8m8tk) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Zephyr DPO 2x faster [free Colab](https://colab.research.google.com/drive/15vttTpzzVXv_tJwEk-hIcQ0S9FcEWvwP?usp=sharing)
2. Llama 7b 2x faster [free Colab](https://colab.research.google.com/drive/1lBzz5KeZJKXjvivbYvmGarix9Ao6Wxe5?usp=sharing)
3. TinyLlama 4x faster full Alpaca 52K in 1 hour [free Colab](https://colab.research.google.com/drive/1AZghoNBQaMDgWJpi4RbffGM1h6raLUj9?usp=sharing)
4. CodeLlama 34b 2x faster [A100 on Colab](https://colab.research.google.com/drive/1y7A0AxE3y8gdj4AVkl2aZX47Xu3P1wJT?usp=sharing)
5. Mistral 7b [free Kaggle version](https://www.kaggle.com/code/danielhanchen/kaggle-mistral-7b-unsloth-notebook)
6. We also did a [blog](https://huggingface.co/blog/unsloth-trl) with 🤗 HuggingFace, and we're in the TRL [docs](https://huggingface.co/docs/trl/main/en/sft_trainer#accelerate-fine-tuning-2x-using-unsloth)!
7. `ChatML` for ShareGPT datasets, [conversational notebook](https://colab.research.google.com/drive/1Aau3lgPzeZKQ-98h69CCu1UJcvIBLmy2?usp=sharing)
8. Text completions like novel writing [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing)
9. [**NEW**] We make Phi-3 Medium / Mini **2x faster**! See our [Phi-3 Medium notebook](https://colab.research.google.com/drive/1hhdhBa1j_hsymiW9m-WzxQtgqTH_NHqi?usp=sharing)
10. [**NEW**] We make Gemma-2 9b / 27b **2x faster**! See our [Gemma-2 9b notebook](https://colab.research.google.com/drive/1vIrqH5uYDQwsJ4-OO3DErvuv4pBgVwk4?usp=sharing)
11. [**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)
12. [**NEW**] We make Mistral NeMo 12B 2x faster and fit in under 12GB of VRAM! [Mistral NeMo notebook](https://colab.research.google.com/drive/17d3U-CAIwzmbDRqbZ9NnpHxCkmXB6LZ0?usp=sharing)

<div class="align-center">
  <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/u54VK8m8tk"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://ko-fi.com/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Kofi button.png" width="145"></a></a> Support our work if you can! Thanks!
</div>

In [32]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [33]:
import os
import glob
import zipfile

# Create a ZIP file
zip_name = "/content/Task_2_Indian_Legal_LoRA.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:

    # 1. Trained LoRA model
    if os.path.exists("/content/lora_model"):
        for root, dirs, files in os.walk("/content/lora_model"):
            for file in files:
                path = os.path.join(root, file)
                zipf.write(path, os.path.relpath(path, "/content"))

    # 2. Training checkpoints
    if os.path.exists("/content/outputs"):
        for root, dirs, files in os.walk("/content/outputs"):
            for file in files:
                path = os.path.join(root, file)
                zipf.write(path, os.path.relpath(path, "/content"))

    # 3. Colab notebook
    notebooks = glob.glob("/content/*.ipynb")
    for notebook in notebooks:
        zipf.write(notebook, os.path.basename(notebook))

print("ZIP created successfully:")
print(zip_name)

ZIP created successfully:
/content/Task_2_Indian_Legal_LoRA.zip


In [34]:
from google.colab import files

files.download("/content/Task_2_Indian_Legal_LoRA.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>